In [2]:
# --- 02_model_baseline.ipynb ---
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/f1-telemetry-ml'
labeled_path = f'{data_path}/labeled'

# CHANGED: paths below now actually use labeled_path (they were still
# hardcoded to the old local '../fastf1_data/...' paths, which don't
# exist in Colab — that's why drive.mount() alone wasn't enough).
train_df = pd.read_parquet(f'{labeled_path}/train_data.parquet')
val_df = pd.read_parquet(f'{labeled_path}/val_data.parquet')
test_in_dist_df = pd.read_parquet(f'{labeled_path}/test_in_dist_data.parquet')
zero_shot_df = pd.read_parquet(f'{labeled_path}/test_zero_shot.parquet')

target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']

# Baseline predictions derived strictly from the Training Set mean
baseline_pred = train_df[target_cols].mean().values

print("=== Mean Baseline Target Values (Train Set) ===")
for col, val in zip(target_cols, baseline_pred):
    print(f"  {col}: {val:.4f}")

def evaluate_split(df: pd.DataFrame, split_name: str):
    preds_matrix = [baseline_pred] * len(df)

    overall_mae = mean_absolute_error(df[target_cols], preds_matrix)
    per_target_mae = mean_absolute_error(df[target_cols], preds_matrix, multioutput='raw_values')

    # CHANGED: added RMSE and R2 (Requirement 9 wants both, not just MAE).
    # ** 0.5 instead of squared=False, since that argument was removed in
    # newer scikit-learn versions — this works on any version.
    overall_rmse = mean_squared_error(df[target_cols], preds_matrix) ** 0.5
    per_target_mse = mean_squared_error(df[target_cols], preds_matrix, multioutput='raw_values')
    per_target_rmse = per_target_mse ** 0.5
    overall_r2 = r2_score(df[target_cols], preds_matrix)
    per_target_r2 = r2_score(df[target_cols], preds_matrix, multioutput='raw_values')

    print(f"\n--- {split_name} Baseline Performance ---")
    print(f"Overall MAE: {overall_mae:.4f}  RMSE: {overall_rmse:.4f}  R2: {overall_r2:.4f}")
    for col, mae_val, rmse_val, r2_val in zip(target_cols, per_target_mae, per_target_rmse, per_target_r2):
        print(f"  {col} — MAE: {mae_val:.4f}  RMSE: {rmse_val:.4f}  R2: {r2_val:.4f}")

evaluate_split(val_df, "Validation Set (Singapore, Austria)")
evaluate_split(test_in_dist_df, "In-Distribution Test Set (Monza, Silverstone Laps)")
evaluate_split(zero_shot_df, "Zero-Shot Holdout Set (Belgium, Jeddah)")

Mounted at /content/drive
=== Mean Baseline Target Values (Train Set) ===
  aggression_score: 0.4488
  line_shape_score: 0.5207
  oversteer_preference_score: 0.4461

--- Validation Set (Singapore, Austria) Baseline Performance ---
Overall MAE: 0.1738  RMSE: 0.2256  R2: -0.0149
  aggression_score — MAE: 0.0923  RMSE: 0.1523  R2: -0.0081
  line_shape_score — MAE: 0.2271  RMSE: 0.2629  R2: -0.0215
  oversteer_preference_score — MAE: 0.2021  RMSE: 0.2458  R2: -0.0152

--- In-Distribution Test Set (Monza, Silverstone Laps) Baseline Performance ---
Overall MAE: 0.2011  RMSE: 0.2537  R2: -0.0154
  aggression_score — MAE: 0.0981  RMSE: 0.1607  R2: -0.0198
  line_shape_score — MAE: 0.2502  RMSE: 0.2913  R2: -0.0216
  oversteer_preference_score — MAE: 0.2551  RMSE: 0.2869  R2: -0.0048

--- Zero-Shot Holdout Set (Belgium, Jeddah) Baseline Performance ---
Overall MAE: 0.2529  RMSE: 0.3002  R2: -0.1980
  aggression_score — MAE: 0.1733  RMSE: 0.2486  R2: -0.2675
  line_shape_score — MAE: 0.2785  RMS